# 📊 Evidencias del Procedimiento y Análisis Estadístico
## Proyecto Historieta: *El Caso "NovaApp / Synapse" y el Poder de la Estadística*

Este cuaderno de **Google Colab** contiene la demostración matemática, simulación de datos, tablas y gráficas que respaldan el desarrollo técnico de las **12 viñetas** de la historieta.

### 📌 Temas del Programa Cubiertos:
* **5.1 Muestreo:**
  * `5.1.1` Tipos de muestreo (Muestreo por conveniencia vs. Muestreo probabilístico estratificado).
  * `5.1.2` Teorema del límite central ($n \ge 30$).
  * `5.1.3` Distribución muestral de la media ($\bar{X}$).
  * `5.1.4` Distribución muestral de una proporción ($\hat{p}$).
* **5.2 Estimaciones:**
  * `5.2.1` Estimación puntual.
  * `5.2.2` Estimación por intervalo.
  * `5.2.3` Intervalo de confianza para una media.
  * `5.2.4` Intervalo de confianza para una proporción.

In [ ]:
# Configuración inicial e importación de librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# Configuración estética de las gráficas
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11

# Semilla aleatoria para reproducibilidad exacta
np.random.seed(42)
print('✅ Librerías cargadas correctamente.')

---
## 1. Simulación de la Población en Crisis: Caso NovaApp / Synapse (Viñetas 1 y 5)
En la **Viñeta 1**, la aplicación **NovaApp** cae a **1.2 estrellas** debido a que el microservicio **Synapse** presenta lentitud y errores críticos tras una actualización.
* **Población total ($N$):** $100,000$ registros activos en el servidor.
* **Estrato 1 — Android ($60\%$):** $N_1 = 60,000$ usuarios (afectados por el bug en la actualización de Android).
* **Estrato 2 — iOS ($40\%$):** $N_2 = 40,000$ usuarios (operando con normalidad).

In [ ]:
# Tamaño de la población total (Censo de reportes/usuarios)
N_total = 100000
prop_android = 0.60
prop_ios = 0.40

N_android = int(N_total * prop_android)  # 60,000 usuarios Android
N_ios = int(N_total * prop_ios)          # 40,000 usuarios iOS

# Simulación del tiempo de carga (en segundos):
# En Android hay alta demora y asimetría por la falla del microservicio Synapse
tiempos_android_pob = np.random.gamma(shape=4.5, scale=0.85, size=N_android) + 0.8
# En iOS el tiempo de carga es rápido y estable (~1.45 s)
tiempos_ios_pob = np.random.normal(loc=1.45, scale=0.30, size=N_ios)
tiempos_ios_pob = np.clip(tiempos_ios_pob, 0.6, None)

# Simulación de fallos críticos (1 = Error en Synapse, 0 = Sin error)
fallos_android_pob = np.random.binomial(1, 0.38, size=N_android)  # 38% de tasa real de error en Android
fallos_ios_pob = np.random.binomial(1, 0.04, size=N_ios)          # 4% de tasa real de error en iOS

# Construcción del DataFrame poblacional
df_poblacion = pd.DataFrame({
    'Sistema_Operativo': ['Android'] * N_android + ['iOS'] * N_ios,
    'Tiempo_Carga_s': np.concatenate([tiempos_android_pob, tiempos_ios_pob]),
    'Fallo_Synapse': np.concatenate([fallos_android_pob, fallos_ios_pob])
})

resumen_pob = df_poblacion.groupby('Sistema_Operativo').agg(
    Usuarios=('Tiempo_Carga_s', 'count'),
    Proporcion_Poblacional=('Tiempo_Carga_s', lambda x: f"{len(x)/N_total*100:.0f}%"),
    Media_Tiempo_s=('Tiempo_Carga_s', 'mean'),
    Desv_Estandar_s=('Tiempo_Carga_s', 'std'),
    Tasa_Fallos_Real=('Fallo_Synapse', lambda x: f"{x.mean()*100:.1f}%")
).reset_index()

print(f" Media poblacional real del tiempo de carga: {df_poblacion['Tiempo_Carga_s'].mean():.2f} segundos")
resumen_pob

---
## 2. El Error del Muestreo por Conveniencia y Estimación Puntual (Viñetas 2, 3 y 4)
**Temas `5.1.1` (Tipos de muestreo) y `5.2.1` (Estimación puntual):**
En las **Viñetas 2 y 3**, Dani toma únicamente $n = 5$ teléfonos de gama alta de la oficina:
$$x = \{1.2,\; 1.5,\; 1.3,\; 1.6,\; 1.4\}\text{ segundos}$$
Obteniendo una **estimación puntual** de $\bar{x} = 1.40\text{ s}$, la cual presenta un fuerte **sesgo de selección** frente a la realidad de los usuarios.

In [ ]:
# Datos medidos por Dani en la Viñeta 2
celulares_oficina = ['Celular 1', 'Celular 2', 'Celular 3', 'Celular 4', 'Celular 5']
tiempos_dani = np.array([1.2, 1.5, 1.3, 1.6, 1.4])
media_dani = np.mean(tiempos_dani)
media_real_pob = df_poblacion['Tiempo_Carga_s'].mean()

print(f"Estimación puntual de Dani (n=5, Muestreo por Conveniencia): {media_dani:.2f} s")
print(f"Media real de toda la población (N=100,000):                 {media_real_pob:.2f} s")
print(f"Sesgo de subestimación:                                     {media_dani - media_real_pob:.2f} s")

# Gráfica 1: Recreación de la gráfica de barras de Dani (Viñeta 3) vs. Realidad Poblacional
fig, ax = plt.subplots(figsize=(9, 5))
colores_dani = ['#e74c3c', '#3498db', '#2ecc71', '#27ae60', '#f1c40f']
barras = ax.bar(celulares_oficina, tiempos_dani, color=colores_dani, edgecolor='black', width=0.55)

ax.axhline(media_dani, color='#2980b9', linestyle='--', linewidth=2, label=f'Promedio de Dani (Conveniencia): {media_dani:.2f} s')
ax.axhline(media_real_pob, color='#c0392b', linestyle='-', linewidth=2.5, label=f'Media Real Poblacional: {media_real_pob:.2f} s (¡Sesgo!)')

for bar in barras:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.08, f'{yval:.1f} s', ha='center', fontweight='bold')

ax.set_ylim(0, 4.2)
ax.set_ylabel('Tiempo de Carga (segundos)')
ax.set_title('Viñetas 2, 3 y 4: Sesgo en el Muestreo por Conveniencia de Dani (n = 5)', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 3. Diseño del Muestreo Probabilístico Estratificado (Viñetas 5 y 6)
**Tema `5.1.1` (Muestreo Estratificado con Afijación Proporcional):**
Para evitar analizar millones de reportes uno por uno (censo) y eliminar el sesgo de selección, Valeria y Mateo dividen la población en dos estratos homogéneos según el sistema operativo, con una muestra total de **$n = 400$ usuarios**:
* **Android ($60\%$):** $n_1 = 0.60 \times 400 = 240$ usuarios.
* **iOS ($40\%$):** $n_2 = 0.40 \times 400 = 160$ usuarios.

In [ ]:
# Parámetros del muestreo estratificado (Viñeta 6)
n_total = 400
n_android = int(n_total * prop_android)  # 240 casos
n_ios = int(n_total * prop_ios)          # 160 casos

# Extracción aleatoria dentro de cada estrato (el script de Dani en la Viñeta 6)
muestra_android = df_poblacion[df_poblacion['Sistema_Operativo'] == 'Android'].sample(n=n_android, random_state=42)
muestra_ios = df_poblacion[df_poblacion['Sistema_Operativo'] == 'iOS'].sample(n=n_ios, random_state=42)
df_muestra_estratificada = pd.concat([muestra_android, muestra_ios], ignore_index=True)

print(f"Muestra total extraída: n = {len(df_muestra_estratificada)} usuarios")
print(f"  • Estrato Android (60%): n1 = {len(muestra_android)} casos")
print(f"  • Estrato iOS (40%):     n2 = {len(muestra_ios)} casos")

# Gráfica 2: Recreación visual de la pantalla de la Viñeta 6
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ax1.pie([prop_android*100, prop_ios*100], labels=['Android (60%)', 'iOS (40%)'],
        autopct='%1.0f%%', startangle=90, colors=['#5dade2', '#82e0aa'],
        wedgeprops={'edgecolor': 'black', 'linewidth': 1.5})
ax1.set_title('Proporción Poblacional por SO (Viñeta 6)', fontweight='bold')

barras_est = ax2.bar(['Android (60%)', 'iOS (40%)'], [n_android, n_ios],
                     color=['#5dade2', '#82e0aa'], edgecolor='black', width=0.5)
for b in barras_est:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 6,
             f'{int(b.get_height())} casos', ha='center', fontweight='bold')
ax2.set_ylim(0, 290)
ax2.set_ylabel('Número de usuarios en la muestra (n = 400)')
ax2.set_title('Asignación Proporcional en la Muestra (n = 400)', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Teorema del Límite Central y Reducción de Varianza / Error Estándar (Viñetas 7, 8 y 11)
**Temas `5.1.2` (Teorema del Límite Central) y `5.1.3` (Distribución muestral de la media):**
1. Como el tamaño de muestra es $n = 400 \ge 30$, por el **Teorema del Límite Central (TLC)** la distribución muestral de la media ($\bar{X}$) deja de ser asimétrica/caótica y se aproxima a una **Distribución Normal** (tal como se observa en el monitor de la **Viñeta 8**).
2. Al separar la variabilidad entre Android e iOS mediante el **Muestreo Aleatorio Estratificado (MAE)** frente a un **Muestreo Aleatorio Simple (MAS)**, la varianza de la estimación disminuye ~45% (**Viñeta 7**) y el error estándar se reduce cerca de un **30%** (**Viñeta 11**):
$$V(\bar{x}_{\text{est}}) = \sum_{h=1}^{L} W_h^2 \frac{s_h^2}{n_h} \quad < \quad V(\bar{x}_{\text{MAS}}) = \frac{s_{\text{total}}^2}{n}$$

In [ ]:
# Cálculo comparativo de Varianza y Error Estándar: MAS vs. Estratificado
s2_total = df_muestra_estratificada['Tiempo_Carga_s'].var(ddof=1)
s2_android = muestra_android['Tiempo_Carga_s'].var(ddof=1)
s2_ios = muestra_ios['Tiempo_Carga_s'].var(ddof=1)

# Varianza y Error Estándar bajo Muestreo Aleatorio Simple (MAS)
var_mas = s2_total / n_total
ee_mas = np.sqrt(var_mas)

# Varianza y Error Estándar bajo Muestreo Estratificado (MAE)
var_est = (prop_android**2 * (s2_android / n_android)) + (prop_ios**2 * (s2_ios / n_ios))
ee_est = np.sqrt(var_est)

reduccion_varianza = (1 - (var_est / var_mas)) * 100
reduccion_ee = (1 - (ee_est / ee_mas)) * 100

print(f"Varianza de la media (Muestreo Simple):        {var_mas:.6f}")
print(f"Varianza de la media (Muestreo Estratificado): {var_est:.6f}")
print(f"📉 Reducción en la Varianza (Viñeta 7):        {reduccion_varianza:.1f}%\n")

print(f"Error Estándar (Muestreo Simple):              {ee_mas:.4f} s")
print(f"Error Estándar (Muestreo Estratificado):       {ee_est:.4f} s")
print(f"📉 Reducción en el Error Estándar (Viñeta 11): {reduccion_ee:.1f}%")

# Gráfica 3: Recreación exacta del monitor de la Viñeta 8 (Datos Crudos Asimétricos vs. Campana Normal por TLC)
medias_muestrales_tlc = [
    df_poblacion['Tiempo_Carga_s'].sample(n=n_total).mean() for _ in range(1500)
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

# Izquierda: Distribución sesgada/caótica de tiempos individuales
ax1.hist(df_poblacion['Tiempo_Carga_s'], bins=35, color='#5499c7', edgecolor='black', alpha=0.85, density=True)
ax1.set_title('Población Original: Tiempos de Respuesta (Asimétrica)', fontweight='bold')
ax1.set_xlabel('Tiempo de Carga (segundos)')
ax1.set_ylabel('Densidad')

# Derecha: Distribución muestral de la media (Campana de Gauss por TLC, n=400 > 30)
ax2.hist(medias_muestrales_tlc, bins=30, color='#5499c7', edgecolor='black', alpha=0.6, density=True, label='Medias muestrales (n=400)')
x_vals = np.linspace(min(medias_muestrales_tlc), max(medias_muestrales_tlc), 200)
ax2.plot(x_vals, stats.norm.pdf(x_vals, np.mean(medias_muestrales_tlc), np.std(medias_muestrales_tlc)),
         color='#1b4f72', linewidth=2.5, label='Curva Normal Teórica (TLC)')
ax2.set_title('Viñeta 8: Distribución Muestral de la Media (TLC, n = 400 > 30)', fontweight='bold')
ax2.set_xlabel('Media Muestral del Tiempo de Carga (s)')
ax2.set_ylabel('Densidad')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 5. Estimación por Intervalo: Media y Proporción de Fallos (Viñetas 7, 9 y 10)
**Temas `5.1.4`, `5.2.2`, `5.2.3` (IC para la media) y `5.2.4` (IC para la proporción):**
* **Intervalo de Confianza al $95\%$ para la Media ($\mu$):**
$$\bar{x} \pm Z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}$$
* **Intervalo de Confianza al $95\%$ para la Proporción de Fallos ($p$):**
$$\hat{p} \pm Z_{\alpha/2} \cdot \sqrt{\frac{\hat{p}(1 - \hat{p})}{n}}$$
En las **Viñetas 7 y 9**, Valeria confirma mediante estos intervalos de confianza que la anomalía es generalizada en la actualización de **Android** y se concentra en el microservicio **Synapse**.

In [ ]:
z_95 = stats.norm.ppf(0.975)  # 1.96 para 95% de confianza

resultados_ic = []
for so, grupo in [('Android', muestra_android), ('iOS', muestra_ios)]:
    n_g = len(grupo)
    # 5.2.3 Intervalo de confianza para la media del tiempo de carga
    media_t = grupo['Tiempo_Carga_s'].mean()
    ee_t = grupo['Tiempo_Carga_s'].std(ddof=1) / np.sqrt(n_g)
    ic_t_inf, ic_t_sup = media_t - z_95 * ee_t, media_t + z_95 * ee_t
    
    # 5.1.4 y 5.2.4 Intervalo de confianza para la proporción de errores en Synapse
    p_hat = grupo['Fallo_Synapse'].mean()
    ee_p = np.sqrt((p_hat * (1 - p_hat)) / n_g)
    ic_p_inf, ic_p_sup = max(0, p_hat - z_95 * ee_p), min(1, p_hat + z_95 * ee_p)
    
    resultados_ic.append({
        'Estrato (SO)': so,
        'Muestra (n)': n_g,
        'Media Tiempo (s)': round(media_t, 3),
        'IC 95% Media (s)': f"[{ic_t_inf:.2f} s, {ic_t_sup:.2f} s]",
        'Proporción Fallos (p̂)': f"{p_hat*100:.1f}%",
        'IC 95% Proporción Fallos': f"[{ic_p_inf*100:.1f}%, {ic_p_sup*100:.1f}%]",
        'p_hat_num': p_hat * 100,
        'margen_p_num': z_95 * ee_p * 100
    })

df_ic = pd.DataFrame(resultados_ic)
display(df_ic[['Estrato (SO)', 'Muestra (n)', 'Media Tiempo (s)', 'IC 95% Media (s)', 'Proporción Fallos (p̂)', 'IC 95% Proporción Fallos']])

# Gráfica 4: Recreación de los monitores de las Viñetas 7, 9 y 10
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

# Subgráfica izquierda: Proporción de fallos con Intervalos de Confianza al 95% (Viñetas 7 y 9)
estratos = df_ic['Estrato (SO)']
props = df_ic['p_hat_num']
errores = df_ic['margen_p_num']
barras_p = ax1.bar(estratos, props, yerr=errores, capsize=8, color=['#e67e22', '#2ecc71'], edgecolor='black', width=0.45)
for b, err in zip(barras_p, errores):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + err + 1.5,
             f"{b.get_height():.1f}% (±{err:.1f}%)", ha='center', fontweight='bold')
ax1.set_ylim(0, 52)
ax1.set_ylabel('Porcentaje de Fallos Críticos (%)')
ax1.set_title('Viñetas 7 y 9: IC 95% para la Proporción de Fallos por SO', fontweight='bold')

# Subgráfica derecha: Incidencia por Microservicio (Monitor de la Viñeta 10 aislando a Synapse)
microservicios = ['Auth', 'Pagos', 'Notificaciones', 'Perfil', 'Synapse (Android)']
tasas_ms = [2.1, 3.4, 1.8, 2.9, props.iloc[0]]
colores_ms = ['#7f8c8d', '#7f8c8d', '#7f8c8d', '#7f8c8d', '#e74c3c']
barras_ms = ax2.bar(microservicios, tasas_ms, color=colores_ms, edgecolor='black', width=0.55)
for b in barras_ms:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 1.0,
             f"{b.get_height():.1f}%", ha='center', fontweight='bold')
ax2.set_ylim(0, 48)
ax2.set_ylabel('Tasa de Error (%)')
ax2.set_title('Viñeta 10: Localización del Microservicio Fallido (Synapse)', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. Estabilización de "Synapse" al 100% tras la Corrección (Viñetas 11 y 12)
Una vez aislado y corregido el fallo en el microservicio **Synapse** para dispositivos Android sin detener la operación general, el equipo valida la estabilidad del sistema (como celebran Dani, Valeria, Mateo y el Sr. Robles en las **Viñetas 11 y 12**).

In [ ]:
# Simulación del monitoreo de estabilidad (Disponibilidad / Eficiencia del servicio Synapse por hora)
horas = ['08:00', '09:00', '10:00', '11:00 (Muestreo)', '12:00 (Parche)', '13:00', '14:00 (Producción)']
estabilidad_synapse = [62.0, 59.5, 61.2, 74.0, 92.5, 98.8, 100.0]

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.bar(horas, estabilidad_synapse, color='#85c1e9', edgecolor='black', alpha=0.7, width=0.5, label='Disponibilidad por corte')
ax.plot(horas, estabilidad_synapse, color='#1b4f72', marker='o', linewidth=2.5, markersize=8, label='Tendencia de recuperación')

for x_i, y_i in zip(horas, estabilidad_synapse):
    ax.text(x_i, y_i + 2.2, f'{y_i:.1f}%', ha='center', fontweight='bold')

ax.set_ylim(0, 112)
ax.set_ylabel('Estabilidad del Microservicio Synapse (%)')
ax.set_title('Viñetas 11 y 12: Recuperación y Estabilidad al 100% Basada en Datos', fontweight='bold')
ax.legend(loc='lower right')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()